# Compare comorbidities from Concomitant Meds and General Physical Exam

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pie_clean import *

In [ ]:
# First, load and clean the data and check we have Concomitant Meds and General Physical Exam
data = DataLoader.load("../PPMI")
cm_df = data[MEDICAL_HISTORY]["Concomitant_Medication"]
print(cm_df.shape)
pe_df = data[MEDICAL_HISTORY]["General_Physical_Exam"]
print(pe_df.shape)

In [ ]:
# Build a table of which comorbidities each patient has ever had, according to meds
meds = cm_df.pivot_table(index="PATNO", columns="CMINDC_TEXT", values="CMINDC",
                         aggfunc=lambda v: 1, fill_value=0)
print(meds.shape)
meds.head()

In [ ]:
# Build a table of which comorbidities each patient has ever had, according to exam
phys = pe_df.pivot_table(index="PATNO", columns="PECAT", values="ABNORM", aggfunc="max", fill_value=0)
print(phys.shape)
phys.head()

In [ ]:
# Merge both together, focusing on patients with both
merge = meds.merge(phys, how="inner", left_index=True, right_index=True)
print(merge.columns.tolist())
merge.head()

In [ ]:
# Let's compare "Psychiatric" on physical exam with "Depression" on meds
pd.crosstab(merge["Psychiatric"], merge["Depression"])

In [ ]:
# Now let's visualize a few more medications, and whether patients were assessed
# as having a Psychiatric condition (No: 0, Yes: 1, Unable to assess: 0.5)
meds_of_interest = ["Anxiety", "Depression", "Delusions, Hallucination, Psychosis"]
tmp = merge.reset_index()[["PATNO", "Psychiatric"]+meds_of_interest].melt(
    id_vars=["PATNO", "Psychiatric"], var_name="On medication for", value_name="Value")
sns.catplot(data=tmp, col="On medication for", hue="Value", kind="count", x="Psychiatric")

We can see that many patients on medication for Anxiety and/or Depression were assessed as having no Psychiatric condition (orange columns at 0.0, left and centre plots). The only orange column larger than its partner blue column is in the centre plot, where those assessed as having a Psychiatric condition are more likely to be on medication for Depression than not on such medication.